In [1]:
# Added for the public reproduction package: load the pseudonymized CSV instead of the
# private spreadsheet and write any output files to ../output/notebooks/. See repro_data.py.
import repro_data


In [2]:
import pandas as pd
from scipy import stats

# ============================================================
# Configuración
# ============================================================

ARCHIVO = "Wallet-pattern-.xlsx"
HOJA = "Datos cuanti"

COL_GRUPO = "Grupo"
COL_SUS = "SUS UX"

TRATAMIENTOS = {
    1: "single-sig-mobile",
    2: "multi-sig-desktop-init",
    3: "multi-sig-mobile-init",
    4: "single-sig-web",
}

# ============================================================
# Funciones
# ============================================================

def cargar_datos(path):
    df = repro_data.read_datos_cuanti()

    df = df[[COL_GRUPO, COL_SUS]].dropna()

    df[COL_GRUPO] = df[COL_GRUPO].astype(int)
    df[COL_SUS] = pd.to_numeric(df[COL_SUS])

    return df


def grupo(df, n):
    return df.loc[df[COL_GRUPO] == n, COL_SUS]


def describir(nombre, serie):
    print(
        f"{nombre:25}"
        f" n={len(serie):2d}"
        f"  media={serie.mean():6.2f}"
        f"  DE={serie.std():6.2f}"
        f"  mediana={serie.median():6.2f}"
    )


def mann_whitney(nombre_test, s_a, etiqueta_a, s_b, etiqueta_b):

    print("\n" + "=" * 70)
    print(nombre_test)
    print("=" * 70)

    describir(etiqueta_a, s_a)
    describir(etiqueta_b, s_b)

    u, p = stats.mannwhitneyu(
        s_a,
        s_b,
        alternative="two-sided",
        method="auto"
    )

    print("\nMann-Whitney U test (two-sided)")
    print(f"U = {u:.2f}")
    print(f"p = {p:.5f}")

    if p < 0.05:
        print("Result: statistically significant difference.")
    else:
        print("Result: no statistically significant difference.")

    return u, p


# ============================================================
# Programa principal
# ============================================================

def main():

    df = cargar_datos(ARCHIVO)

    print("Conteo por grupo")
    print("-" * 40)
    print(df[COL_GRUPO].value_counts().sort_index())

    t1 = grupo(df, 1)
    t2 = grupo(df, 2)
    t3 = grupo(df, 3)
    t4 = grupo(df, 4)

    # --------------------------------------------------------
    # RQ2 - Single-sign
    # T1 vs T4
    # --------------------------------------------------------

    u_single, p_single = mann_whitney(
        "RQ2 - Single-sign (T1 vs T4)",
        t1,
        TRATAMIENTOS[1],
        t4,
        TRATAMIENTOS[4]
    )

    # --------------------------------------------------------
    # RQ2 - Multi-sign
    # T2 vs T3
    # --------------------------------------------------------

    u_multi, p_multi = mann_whitney(
        "RQ2 - Multi-sign (T2 vs T3)",
        t2,
        TRATAMIENTOS[2],
        t3,
        TRATAMIENTOS[3]
    )

    print("\n" + "=" * 70)
    print("SUMMARY")
    print("=" * 70)

    print(f"T1 vs T4 : U = {u_single:.2f}, p = {p_single:.5f}")

    print(f"T2 vs T3 : U = {u_multi:.2f}, p = {p_multi:.5f}")

    print("\nInterpretation")

    if p_single < 0.05:
        print("• T1 vs T4: statistically significant difference.")
    else:
        print("• T1 vs T4: no statistically significant difference.")

    if p_multi < 0.05:
        print("• T2 vs T3: statistically significant difference.")
    else:
        print("• T2 vs T3: no statistically significant difference.")


main()

Conteo por grupo
----------------------------------------
Grupo
1    17
2    17
3    18
4    15
Name: count, dtype: int64

RQ2 - Single-sign (T1 vs T4)
single-sig-mobile         n=17  media= 76.76  DE= 15.98  mediana= 77.50
single-sig-web            n=15  media= 77.67  DE= 11.44  mediana= 77.50

Mann-Whitney U test (two-sided)
U = 130.00
p = 0.93955
Result: no statistically significant difference.

RQ2 - Multi-sign (T2 vs T3)
multi-sig-desktop-init    n=17  media= 62.65  DE= 22.99  mediana= 72.50
multi-sig-mobile-init     n=18  media= 73.89  DE= 12.90  mediana= 71.25

Mann-Whitney U test (two-sided)
U = 119.50
p = 0.27507
Result: no statistically significant difference.

SUMMARY
T1 vs T4 : U = 130.00, p = 0.93955
T2 vs T3 : U = 119.50, p = 0.27507

Interpretation
• T1 vs T4: no statistically significant difference.
• T2 vs T3: no statistically significant difference.
